In [ ]:
%%sql -r dataframe_2
-- Active riders the company has this month
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW ACTIVE_RIDERS AS

SELECT DISTINCT
   rider_id,
FROM database_name.schema_name.silver_orders
WHERE delivery_time >= DATEADD('day', -28, CURRENT_DATE())


In [ ]:
%%sql -r dataframe_10
-- Active riders the company had last month
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW ACTIVE_RIDERS_PREV_MONTH AS

SELECT DISTINCT
    rider_id,
FROM database_name.schema_name.silver_orders
WHERE delivery_time >= DATEADD('day', -56, CURRENT_DATE()) AND delivery_time < DATEADD('day', -28, CURRENT_DATE())

In [ ]:
%%sql -r dataframe_5
-- Power riders the company has this week
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW POWER_RIDERS AS

SELECT
    rider_id,
FROM database_name.schema_name.silver_orders
WHERE delivery_time >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY rider_id
HAVING COUNT(order_id) >= 7

In [ ]:
%%sql -r dataframe_11
-- Power riders the company had last week
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW POWER_RIDERS_PREV_WEEK AS

SELECT
    rider_id,
FROM database_name.schema_name.silver_orders
WHERE delivery_time >= DATEADD('day', -14, CURRENT_DATE()) AND delivery_time < DATEADD('day', -7, CURRENT_DATE())
GROUP BY rider_id
HAVING COUNT(order_id) >= 7

In [ ]:
%%sql -r dataframe_3
-- Who are the new riders that have joined over the last week?
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW NEW_RIDERS AS

SELECT
    rider_id,
    TRIM(CONCAT(first_name, ' ', last_name)) AS full_name,
    join_date
FROM database_name.schema_name.silver_rider
WHERE join_date >= DATEADD('day', -7, CURRENT_DATE())

In [ ]:
%%sql -r dataframe_12
-- Who are the new riders that have joined over the week before the last week?
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW NEW_RIDERS_PREV_WEEK AS

SELECT
    rider_id,
    TRIM(CONCAT(first_name, ' ', last_name)) AS full_name,
    join_date
FROM database_name.schema_name.silver_rider
WHERE join_date >= DATEADD('day', -14, CURRENT_DATE()) AND join_date < DATEADD('day', -7, CURRENT_DATE())

In [ ]:
%%sql -r dataframe_8
-- Most Deliveries Award: Top 3 riders (with ties) that have delivered the most parcels in the last year.
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW MOST_DELIVERIES_AWARD AS

SELECT
    o.rider_id,
    TRIM(CONCAT(r.first_name, ' ', r.last_name)) AS full_name,
    COUNT(o.order_id) AS delivery_count,
    DENSE_RANK() OVER (ORDER BY COUNT(o.order_id) DESC) AS rank
FROM database_name.schema_name.silver_orders AS o
LEFT JOIN database_name.schema_name.silver_rider AS r
ON o.rider_id = r.rider_id
WHERE o.delivery_time >= DATEADD('day', -365, CURRENT_DATE())
GROUP BY o.rider_id, r.first_name, r.last_name
QUALIFY DENSE_RANK() OVER (ORDER BY COUNT(o.order_id) DESC) <= 3
ORDER BY COUNT(o.order_id) DESC;

In [ ]:
%%sql -r dataframe_4
-- Big Day Award: Top 3 riders (with ties) that have delivered the most parcels in one day over the last year.
-- Ties are broken by the highest total mileage on that day.
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW BIG_DAY_AWARD AS

SELECT
    o.rider_id,
    TRIM(CONCAT(ri.first_name, ' ', ri.last_name)) AS full_name,
    DATE(o.delivery_time) AS date,
    COUNT(o.order_id) AS delivery_count,
    ROUND(SUM(
        HAVERSINE(
            STRTOK(o.delivery_coords, ',', 1)::FLOAT,
            STRTOK(o.delivery_coords, ',', 2)::FLOAT,
            STRTOK(re.complete_coords, ',', 1)::FLOAT,
            STRTOK(re.complete_coords, ',', 2)::FLOAT
        ) * 0.621371 -- converts to miles
    ), 2) AS total_miles_travelled,
    DENSE_RANK() OVER (ORDER BY COUNT(o.order_id) DESC, ROUND(SUM(
        HAVERSINE(
            STRTOK(o.delivery_coords, ',', 1)::FLOAT,
            STRTOK(o.delivery_coords, ',', 2)::FLOAT,
            STRTOK(re.complete_coords, ',', 1)::FLOAT,
            STRTOK(re.complete_coords, ',', 2)::FLOAT
        ) * 0.621371 -- converts to miles
    ), 2) DESC) AS rank
FROM database_name.schema_name.silver_orders AS o
LEFT JOIN database_name.schema_name.silver_restaurant AS re
    ON o.restaurant_id = re.restaurant_id
LEFT JOIN database_name.schema_name.silver_rider AS ri
    ON o.rider_id = ri.rider_id
WHERE o.delivery_time >= DATEADD('day', -365, CURRENT_DATE())
GROUP BY DATE(o.delivery_time), o.rider_id, ri.first_name, ri.last_name
QUALIFY DENSE_RANK() OVER (ORDER BY COUNT(o.order_id) DESC, ROUND(SUM(
        HAVERSINE(
            STRTOK(o.delivery_coords, ',', 1)::FLOAT,
            STRTOK(o.delivery_coords, ',', 2)::FLOAT,
            STRTOK(re.complete_coords, ',', 1)::FLOAT,
            STRTOK(re.complete_coords, ',', 2)::FLOAT
        ) * 0.621371 -- converts to miles
    ), 2) DESC) <= 3;

In [ ]:
%%sql -r dataframe_1
-- Longest Serving Award: Top 3 riders (with ties) with the earliest join date who are currently active.
-- Ties are broken by the highest total number of deliveries.
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW LONGEST_SERVING_AWARD AS

WITH tiedawards AS (
SELECT
    o.rider_id,
    TRIM(CONCAT(r.first_name, ' ', r.last_name)) AS full_name,
    r.join_date
FROM database_name.schema_name.silver_orders AS o
LEFT JOIN database_name.schema_name.silver_rider AS r
ON o.rider_id = r.rider_id
WHERE o.delivery_time >= DATEADD('day', -28, CURRENT_DATE())
GROUP BY o.rider_id, r.first_name, r.last_name, r.join_date
)
SELECT
    t.rider_id,
    t.full_name,
    t.join_date,
    COUNT(o.order_id) AS total_deliveries,
    DENSE_RANK() OVER (ORDER BY t.join_date ASC, total_deliveries DESC) AS rank
FROM tiedawards AS t
LEFT JOIN database_name.schema_name.silver_orders AS o
ON t.rider_id = o.rider_id
GROUP BY t.rider_id, t.full_name, t.join_date
QUALIFY DENSE_RANK() OVER (ORDER BY t.join_date ASC, total_deliveries DESC) <= 3

In [ ]:
%%sql -r dataframe_9
-- Most Travelled Award: Rider who has put in the most miles over the last year.
USE DATABASE database_name;
USE SCHEMA schema_name;

CREATE OR REPLACE VIEW MOST_TRAVELLED_AWARD AS

SELECT
    o.rider_id,
    TRIM(CONCAT(ri.first_name, ' ', ri.last_name)) AS full_name,
    ROUND(SUM(
        HAVERSINE(
            STRTOK(o.delivery_coords, ',', 1)::FLOAT,
            STRTOK(o.delivery_coords, ',', 2)::FLOAT,
            STRTOK(re.complete_coords, ',', 1)::FLOAT,
            STRTOK(re.complete_coords, ',', 2)::FLOAT
        ) * 0.621371 -- converts to miles
    ), 2) AS total_miles_travelled,
    DENSE_RANK() OVER (ORDER BY ROUND(SUM(
        HAVERSINE(
            STRTOK(o.delivery_coords, ',', 1)::FLOAT,
            STRTOK(o.delivery_coords, ',', 2)::FLOAT,
            STRTOK(re.complete_coords, ',', 1)::FLOAT,
            STRTOK(re.complete_coords, ',', 2)::FLOAT
        ) * 0.621371 -- converts to miles
    ), 2) DESC) AS rank
FROM database_name.schema_name.silver_orders AS o
LEFT JOIN database_name.schema_name.silver_restaurant AS re
    ON o.restaurant_id = re.restaurant_id
LEFT JOIN database_name.schema_name.silver_rider AS ri
    ON o.rider_id = ri.rider_id
WHERE o.delivery_time >= DATEADD('day', -365, CURRENT_DATE())
GROUP BY o.rider_id, ri.first_name, ri.last_name
QUALIFY DENSE_RANK() OVER (ORDER BY ROUND(SUM(
        HAVERSINE(
            STRTOK(o.delivery_coords, ',', 1)::FLOAT,
            STRTOK(o.delivery_coords, ',', 2)::FLOAT,
            STRTOK(re.complete_coords, ',', 1)::FLOAT,
            STRTOK(re.complete_coords, ',', 2)::FLOAT
        ) * 0.621371 -- converts to miles
    ), 2) DESC) <= 3
ORDER BY total_miles_travelled DESC;
